# R3maJ — Colab Training + W&B

Full v3 workflow restored: GitHub clone, Google Drive mount/restore, replay restore, reward-curriculum sanity check, build, W&B training metrics, system metrics, checkpoint backup, crash restart, status and clean stop.

**Before running:** select a T4 GPU and add `WANDB_API_KEY` in Colab Secrets. Never hard-code the key.

In [1]:
# 1. Clone / update R3maJ
import os, subprocess
ROOT = '/content/R3maJ'
REPO = 'https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT], check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'], check=False)
print('ROOT:', ROOT)
print('contents:', sorted(os.listdir(ROOT)))

ROOT: /content/R3maJ
contents: ['.git', '.gitignore', 'CMakeLists.txt', 'R3maJ_WandB_Colab.ipynb', 'R3maJ_colab.ipynb', 'R3maJ_colab_v3.ipynb', 'collision_meshes', 'src', 'thirdparty']


In [2]:
# 2. Mount Google Drive EARLY + restore latest checkpoint + replay binary
import os, shutil, glob
if not os.path.isdir('/content/drive'):
    print('mounting Drive...', flush=True)
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('drive already mounted', flush=True)
DRIVE_CKPT='/content/drive/MyDrive/R3maJ/checkpoints'
LOCAL_CKPT='/content/R3maJ/build/checkpoints'
DRIVE_REPLAY='/content/drive/MyDrive/R3maJ/serialized_replays.bin'
LOCAL_REPLAY='/content/R3maJ/build/serialized_replays.bin'
os.makedirs(LOCAL_CKPT, exist_ok=True)
def ts_of(d):
    try: return int(os.path.basename(d))
    except Exception: return -1
if os.path.isdir(DRIVE_CKPT):
    dirs=sorted([d for d in glob.glob(os.path.join(DRIVE_CKPT,'*')) if os.path.isdir(d)], key=ts_of)
    print(f'Drive has {len(dirs)} checkpoint dirs')
    if dirs:
        newest=dirs[-1]; dest=os.path.join(LOCAL_CKPT,os.path.basename(newest))
        if not os.path.isdir(dest):
            print('restoring latest checkpoint:', os.path.basename(newest))
            shutil.copytree(newest,dest)
else: print('no Drive checkpoints yet')
if os.path.exists(DRIVE_REPLAY) and not os.path.exists(LOCAL_REPLAY):
    print('copying replay binary from Drive')
    shutil.copy(DRIVE_REPLAY,LOCAL_REPLAY)
print('local checkpoints:', sorted(os.listdir(LOCAL_CKPT)))
print('replay binary present:', os.path.exists(LOCAL_REPLAY))

mounting Drive...
Mounted at /content/drive
Drive has 0 checkpoint dirs
copying replay binary from Drive
local checkpoints: []
replay binary present: True


In [3]:
# 3. Curriculum sanity check
import os
def has(path, needle, label):
    ok=os.path.exists(path) and needle in open(path,errors='ignore').read()
    print(('OK  ' if ok else 'MISS ')+label)
pm=os.path.join(ROOT,'src','PhaseManager.cpp')
main=os.path.join(ROOT,'src','main.cpp')
has(pm,'MECH_RAMP_END','mechanical ramp')
has(pm,'PhaseManager::PhaseManager()','phase table')
has(main,'GetRewards(g_totalTimesteps)','auto phase selection')
print('Curriculum check complete.')

OK  mechanical ramp
OK  phase table
OK  auto phase selection
Curriculum check complete.


In [4]:
# W&B setup — Colab Secret
import os
import subprocess

from google.colab import userdata

# Install/update W&B first
subprocess.run(
    ["pip", "install", "-q", "-U", "wandb"],
    check=True
)

import wandb

# Load API key from Colab Secret
WANDB_API_KEY = userdata.get("WANDB_API_KEY")

if not WANDB_API_KEY:
    raise RuntimeError(
        "WANDB_API_KEY not found. Add it under Colab -> Secrets."
    )

# Explicitly authenticate W&B
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_MODE"] = "online"

login_result = wandb.login(
    key=WANDB_API_KEY,
    relogin=True
)

if not login_result:
    raise RuntimeError("W&B login failed.")

print("W&B login: OK")

# Start run
run = wandb.init(
    project="R3maJ",
    name="R3maJ-training",
    config={
        "architecture": "512x6",
        "games": 164,
        "device": "cuda",
        "tick_skip": 8,
        "action_delay": 2,
    },
)

print("wandb:", wandb.__version__)
print("W&B run:", run.url)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


AuthenticationError: Failed to authenticate with https://api.wandb.ai: invalid credentials: the server did not recognize an account for the configured credentials

In [ ]:
# 5. Training metric parser + W&B logger
import re, threading, time
latest_metrics={}
PATTERNS={
 'train/step_reward':r'Average Step Reward\s*: ?\s*([0-9.eE+-]+)',
 'train/policy_entropy':r'Policy Entropy\s*: ?\s*([0-9.eE+-]+)',
 'train/clipped_reward':r'Clipped Reward Portion\s*: ?\s*([0-9.eE+-]+)',
 'train/total_timesteps':r'Total Timesteps\s*: ?\s*([0-9.eE+-]+)',
 'train/iteration':r'Total Iterations\s*: ?\s*([0-9.eE+-]+)',
 'performance/collection_sps':r'Collection Steps/Second\s*: ?\s*([0-9.eE+-]+)',
 'performance/consumption_sps':r'Consumption Steps/Second\s*: ?\s*([0-9.eE+-]+)',
 'performance/collection_time':r'Collection Time\s*: ?\s*([0-9.eE+-]+)',
 'performance/consumption_time':r'Consumption Time\s*: ?\s*([0-9.eE+-]+)',
 'ball/angular_speed':r'Ball/AngularSpeed\s*: ?\s*([0-9.eE+-]+)',
 'ball/height':r'Ball/Height\s*: ?\s*([0-9.eE+-]+)',
 'ball/speed':r'Ball/Speed\s*: ?\s*([0-9.eE+-]+)'}
def parse_training_line(line):
    for key,pat in PATTERNS.items():
        m=re.search(pat,line)
        if m: latest_metrics[key]=float(m.group(1))
def get_phase(t):
    if t<5_000_000_000:return 0
    if t<15_000_000_000:return 1
    if t<30_000_000_000:return 2
    return 3
def wandb_logger():
    last=-1
    while True:
        time.sleep(5)
        if not latest_metrics: continue
        m=dict(latest_metrics); step=m.get('train/total_timesteps')
        if step is None or int(step)<=last: continue
        step=int(step); last=step; m['curriculum/phase']=get_phase(step)
        try: wandb.log(m,step=step)
        except Exception as e: print('[wandb] log error:',e,flush=True)
if '_R3MAJ_WANDB_LOGGER_' not in globals():
    globals()['_R3MAJ_WANDB_LOGGER_']=True
    threading.Thread(target=wandb_logger,daemon=True).start()
print('W&B training logger started.')

In [ ]:
# 6. Install build dependencies
import subprocess, os, torch
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=True)
print('torch:',torch.__version__,'cuda:',torch.cuda.is_available(),'cuda build:',torch.version.cuda)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# 7. Configure CMake
import os, subprocess, torch
os.chdir(ROOT)
torch_prefix=os.path.dirname(torch.__file__)
r=subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={torch_prefix}'],capture_output=True,text=True)
print('configure rc:',r.returncode)
print((r.stdout+r.stderr)[-4000:])
if r.returncode!=0: raise RuntimeError('CMake configure failed')

In [ ]:
# 8. Build R3maJ
import os, subprocess
nproc=os.cpu_count() or 2
print(f'Building with -j{nproc} ...',flush=True)
r=subprocess.run(['cmake','--build','build','-j',str(nproc)],capture_output=True,text=True)
print('build rc:',r.returncode)
print((r.stdout+r.stderr)[-5000:])
exe=os.path.join(ROOT,'build','R3maJ')
print('binary exists:',os.path.exists(exe))
if r.returncode!=0 or not os.path.exists(exe): raise RuntimeError('Build failed')

In [ ]:
# 9. Environment check
import sys,platform,torch
print('python:',sys.version.split()[0])
print('platform:',platform.platform())
print('torch:',torch.__version__,'cuda:',torch.cuda.is_available())
if torch.cuda.is_available(): print('gpu:',torch.cuda.get_device_name(0))

In [ ]:
# 10. System performance -> W&B (same Python run)
import subprocess, time, threading, psutil
def system_monitor():
    while True:
        d={}
        try:
            out=subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5).stdout.strip()
            v=[float(x.strip()) for x in out.split(',')]; d.update({'sys/gpu_util':v[0],'sys/vram_used_mb':v[1],'sys/vram_total_mb':v[2],'sys/gpu_temp_c':v[3]})
        except Exception: pass
        try:
            d.update({'sys/cpu_util':psutil.cpu_percent(interval=1),'sys/ram_used_gb':round(psutil.virtual_memory().used/1e9,2),'sys/ram_total_gb':round(psutil.virtual_memory().total/1e9,2)})
        except Exception: pass
        if d:
            try: wandb.log(d)
            except Exception: pass
        time.sleep(10)
if '_R3MAJ_SYSMON_' not in globals():
    globals()['_R3MAJ_SYSMON_']=True
    threading.Thread(target=system_monitor,daemon=True).start()
print('system monitor started')

In [ ]:
# 11. Training supervisor + Drive backup + live W&B parser
import os,sys,subprocess,torch,time,threading,shutil,glob
BUILD='/content/R3maJ/build'; LOCAL_CKPT=os.path.join(BUILD,'checkpoints'); DRIVE_CKPT='/content/drive/MyDrive/R3maJ/checkpoints'
os.makedirs(LOCAL_CKPT,exist_ok=True); os.makedirs(DRIVE_CKPT,exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('GPU detected:' if DEVICE=='cuda' else 'WARNING: CPU', torch.cuda.get_device_name(0) if DEVICE=='cuda' else '',flush=True)
def _ts(d):
    try:return int(os.path.basename(d))
    except:return -1
def _backup_once():
    local=sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT,'*')) if os.path.isdir(d)],key=_ts); drive=set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT,'*')))
    for d in local:
        name=os.path.basename(d)
        if name in drive:continue
        tmp=os.path.join(DRIVE_CKPT,name+'.tmp')
        try:
            print(f'[backup] uploading checkpoint {name}...',flush=True); shutil.copytree(d,tmp); shutil.move(tmp,os.path.join(DRIVE_CKPT,name)); print(f'[backup] {name} uploaded',flush=True)
        except Exception as e: print('[backup] err:',e,flush=True); shutil.rmtree(tmp,ignore_errors=True)
def _backup_loop():
    while True:
        try:
            if os.path.isdir('/content/drive'):_backup_once()
        except Exception as e:print('[backup] loop err:',e,flush=True)
        time.sleep(60)
if '_R3MAJ_BACKUP_DAEMON_' not in globals():
    globals()['_R3MAJ_BACKUP_DAEMON_']=True; threading.Thread(target=_backup_loop,daemon=True).start(); print('[backup] 24/7 daemon started',flush=True)
_tailer_lock={'run':True}
if '_R3MAJ_TAILER_' not in globals():
    globals()['_R3MAJ_TAILER_']=True
    def _tail_loop():
        path=os.path.join(BUILD,'train.log')
        while not os.path.exists(path):time.sleep(1)
        with open(path,'r',errors='ignore') as f:
            f.seek(0,os.SEEK_END)
            while True:
                data=f.read()
                if data:
                    sys.stdout.write(data);sys.stdout.flush()
                    for line in data.splitlines():parse_training_line(line)
                else:
                    if not _tailer_lock.get('run',True):return
                    time.sleep(.5)
    threading.Thread(target=_tail_loop,daemon=True).start(); print('[tailer] streaming binary output + W&B metrics...',flush=True)
os.chdir(BUILD)
REPLAY_ARG=['--replays','serialized_replays.bin'] if os.path.exists('serialized_replays.bin') else []
BASE_CMD=['stdbuf','-oL','-eL','./R3maJ','--device',DEVICE,'--phase','-1','--save-dir','checkpoints','--games','164']+REPLAY_ARG
print('BASE CMD:',' '.join(BASE_CMD),flush=True)
backoff=10;attempt=0
with open('train.log','a') as log:
    while True:
        attempt+=1;t0=time.time();print(f'[supervisor] launch #{attempt}',flush=True)
        proc=subprocess.Popen(BASE_CMD,stdout=log,stderr=subprocess.STDOUT,start_new_session=True);rc=proc.wait();elapsed=time.time()-t0
        if rc==0:print('[supervisor] clean exit (rc=0)',flush=True);break
        print(f'[supervisor] crashed rc={rc} after {elapsed:.0f}s — restarting in {backoff}s',flush=True);time.sleep(backoff);backoff=min(backoff*2,600) if elapsed<60 else 10
print('supervisor exited.')

In [ ]:
# 12. STATUS
import os,subprocess,glob
print('drive mounted:',os.path.isdir('/content/drive'))
ck='/content/drive/MyDrive/R3maJ/checkpoints'
print('drive ckpt dirs:',sorted(os.path.basename(d) for d in glob.glob(ck+'/*')) if os.path.isdir(ck) else 'none')
local='/content/R3maJ/build/checkpoints'
print('local ckpt dirs:',sorted(os.path.basename(d) for d in glob.glob(local+'/*')) if os.path.isdir(local) else 'none')
r=subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True);print('R3maJ processes:',r.stdout.strip() or 'none')

In [ ]:
# 13. STOP cleanly
import subprocess,time
r=subprocess.run(['pkill','-TERM','-f','R3maJ'],capture_output=True,text=True);print('SIGTERM sent:',r.returncode==0)
time.sleep(20)
r2=subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True);alive=r2.stdout.strip()
if alive:
    print('still alive, force killing:',alive);subprocess.run(['pkill','-9','-f','R3maJ'],capture_output=True);time.sleep(3)
r3=subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True);print('remaining:',r3.stdout.strip() or 'none')